# 05 Silver DLT Pipeline — Integration-laag

**Laag:** Silver / Integration  
**Schema:** `INTEGRATION`  
**Pipeline-type:** Lakeflow Declarative Pipeline (DLT)

## Wat doet deze pipeline?

Leest van Bronze (`STAGING_AZURESTORAGE`) via **Change Data Feed** en mergt de
change-events declaratief via **`apply_changes`** naar Silver Streaming Tables.
Bronze-overschrijvingen (full-load mode) verschijnen als `delete_row` + `insert_row`-
events in de CDF en worden clean verwerkt — geen Silver-duplicaten, geen pipeline-break
bij een mode-switch in Bronze.  Zie ADR 0002.

De `sales_line` Materialised View joint de twee gecleande Silver-tabellen in batch
(MV-semantiek via `spark.read`) — een MV is hier eenvoudiger dan een
stream-stream-join.

## Tabellen in deze pipeline (slices #18 + #19 + #20 + #21)

| Tabel | Type | Inhoud |
|---|---|---|
| `INTEGRATION.order_header` | Streaming Table | Gecleande `order_header` — type-fixes + snake_case + warn-Expectations + drop-regelfilter |
| `INTEGRATION.order_header_quarantine` | Streaming Table | Rijen die ten minste één drop-regel schenden, met `failed_rules ARRAY<STRING>` |
| `INTEGRATION.order_detail` | Streaming Table | Gecleande `order_detail` — type-fixes + snake_case + warn-Expectations + drop-regelfilter |
| `INTEGRATION.order_detail_quarantine` | Streaming Table | Rijen die ten minste één drop-regel schenden, met `failed_rules ARRAY<STRING>` |
| `INTEGRATION.sales_line` | Materialised View | Geïntegreerde view: `order_header ⨝ order_detail` op `order_id`, één rij per orderregel |

## Type-fixes (Bronze → Silver)

### order_header

| Bronze-kolom | Bronze-type | Silver-kolom | Silver-type |
|---|---|---|---|
| `SERVED_TS` | `StringType` | `served_ts` | `TimestampType` |
| `ORDER_TAX_AMOUNT` | `StringType` | `order_tax_amount` | `DecimalType(38,4)` |
| `ORDER_DISCOUNT_AMOUNT` | `StringType` | `order_discount_amount` | `DecimalType(38,4)` |
| `LOCATION_ID` | `DoubleType` | `location_id` | `DecimalType(38,0)` |
| `DISCOUNT_ID` | `StringType` | `discount_id` | `DecimalType(38,0)` nullable |
| `SHIFT_START_TIME` | `IntegerType` (millis) | `shift_start_time` | `StringType` `'HH:mm:ss'` |
| `SHIFT_END_TIME` | `IntegerType` (millis) | `shift_end_time` | `StringType` `'HH:mm:ss'` |

### order_detail

| Bronze-kolom | Bronze-type | Silver-kolom | Silver-type |
|---|---|---|---|
| `ORDER_ITEM_DISCOUNT_AMOUNT` | `StringType` | `order_item_discount_amount` | `DecimalType(38,4)` |

Alle andere kolommen: zelfde type, hernoemd naar snake_case.
Alle vijf Bronze-auditkolommen worden ongewijzigd doorgegeven.

## Severity-model

- **fail** — `@expect_all_or_fail`; pipeline halt op schending. Toegepast op de
  type-fixed Bronze CDF-view zodat schendingen worden gevangen vóór clean/quarantine-routering.
- **drop** — rij wordt uit de cleansed-tak gefilterd en geroute naar
  `_quarantine` met een `failed_rules`-array.
- **warn** — `@expect_all`; rij blijft in cleansed, schending verschijnt in het DLT-
  eventlog en de pipeline-metrics.

In [ ]:
%run ./_lib/type_helpers

In [ ]:
%run ./_lib/bronze_cdf

In [ ]:
%run ./_lib/rule_engine

In [ ]:
import dlt
from pyspark.sql import functions as F

## Parameters

DLT leest pipeline-parameters uit `spark.conf`.  De `catalog`-parameter wordt doorgegeven
door de DAB-pipelineconfiguratie (`resources/dlt_integration.yml`).

In [ ]:
catalog = spark.conf.get("pipeline.catalog", "DEMO")
bronze_schema = "STAGING_AZURESTORAGE"

print(f"Catalog : {catalog}")
print(f"Bronze  : {catalog}.{bronze_schema}")

## Kwaliteitsregels — order_header

Drie ernstniveaus:

- **fail** — `@expect_all_or_fail`; pipeline halt op schending. Gekoppeld aan de
  type-fixed Bronze CDF-view (vangt schendingen ongeacht clean/quarantine-routering).
- **drop** — rij wordt uit de cleansed-tak gefilterd en geroute naar
  `order_header_quarantine` met een `failed_rules`-array.
- **warn** — `@expect_all`; rij blijft in cleansed, schending verschijnt in DLT-
  event-metrics. Gekoppeld aan de cleansed-tak-view.

Regelnamen volgen het `<column>_<predicate>`-patroon zodat `failed_rules` zelfbeschrijvend
is wanneer een analist de quarantine-tabel bevraagt.

In [ ]:
ORDER_HEADER_FAIL_RULES = {
    "order_id_not_null":         "order_id IS NOT NULL",
}

ORDER_HEADER_DROP_RULES = {
    "order_ts_not_null":         "order_ts IS NOT NULL",
    "customer_id_not_null":      "customer_id IS NOT NULL",
    "order_currency_not_null":   "order_currency IS NOT NULL",
    "order_total_non_negative":  "order_total >= 0",
    "order_amount_non_negative": "order_amount >= 0",
}

ORDER_HEADER_WARN_RULES = {
    "truck_id_not_null":         "truck_id IS NOT NULL",
    "location_id_not_null":      "location_id IS NOT NULL",
    "shift_times_ordered":       "shift_start_time <= shift_end_time",
}

ORDER_HEADER_CLEAN_PREDICATE = build_clean_predicate(ORDER_HEADER_DROP_RULES)

print(f"clean-predicaat: {ORDER_HEADER_CLEAN_PREDICATE}")

## order_header — gecleande Streaming Table

**Patroon:**
1. `dlt.create_streaming_table` declareert het doel met `apply_changes`-semantiek.
2. Een `@dlt.view` leest Bronze CDF en past type-fixes + snake_case-rename toe.
3. `dlt.apply_changes` koppelt de view aan de Streaming Table-doeltabel.

De CDF `_change_type`-kolom stuurt de MERGE: `insert_row` / `update_postimage`
worden upserts; `delete_row` / `update_preimage` veroorzaken deletes.  `truncate_row`-
events (uit full-load-overwrites) worden door `apply_changes` als deletes behandeld.

In [ ]:
dlt.create_streaming_table(
    name="order_header",
    comment="Cleansed order_header: type-fixes, snake_case columns, CDF+apply_changes from Bronze.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_header_bronze_cdf")
@dlt.expect_all_or_fail(ORDER_HEADER_FAIL_RULES)
def order_header_bronze_cdf():
    """Lees Bronze order_header via CDF en pas type-fixes + snake_case-rename toe.

    Geeft een streaming DataFrame terug met:
    - Alle bedrijfskolommen hernoemd naar snake_case met type-correcties
    - Alle vijf Bronze-auditkolommen ongewijzigd doorgegeven
    - CDF-metadatakolommen (_change_type, _commit_version, _commit_timestamp)
      bewaard voor apply_changes-routering

    De fail-regel-Expectation halt de pipeline op `order_id IS NULL`, ongeacht in
    welke downstream-tak (clean / quarantine) de rij anders zou belanden.
    """
    src = read_bronze_cdf(f"{catalog}.{bronze_schema}.order_header")

    return (
        src
        # ----------------------------------------------------------------
        # Bedrijfskolommen — type-fixes + snake_case
        # ----------------------------------------------------------------
        .withColumn("order_id",               F.col("ORDER_ID").cast("decimal(38,0)"))
        .withColumn("truck_id",               F.col("TRUCK_ID").cast("decimal(38,0)"))
        .withColumn("location_id",            cast_decimal_38_0(F.col("LOCATION_ID")))   # Double → Decimal
        .withColumn("customer_id",            F.col("CUSTOMER_ID").cast("decimal(38,0)"))
        .withColumn("discount_id",            cast_decimal_38_0(F.col("DISCOUNT_ID")))   # String → Decimal nullable
        .withColumn("shift_id",               F.col("SHIFT_ID").cast("decimal(38,0)"))
        .withColumn("shift_start_time",       millis_to_hhmmss(F.col("SHIFT_START_TIME")))  # Int millis → HH:mm:ss
        .withColumn("shift_end_time",         millis_to_hhmmss(F.col("SHIFT_END_TIME")))    # Int millis → HH:mm:ss
        .withColumn("order_channel",          F.col("ORDER_CHANNEL"))
        .withColumn("order_ts",               F.col("ORDER_TS"))
        .withColumn("served_ts",              cast_served_ts(F.col("SERVED_TS")))            # String → Timestamp
        .withColumn("order_currency",         F.col("ORDER_CURRENCY"))
        .withColumn("order_amount",           F.col("ORDER_AMOUNT").cast("decimal(38,4)"))
        .withColumn("order_tax_amount",       cast_decimal_38_4(F.col("ORDER_TAX_AMOUNT")))        # String → Decimal
        .withColumn("order_discount_amount",  cast_decimal_38_4(F.col("ORDER_DISCOUNT_AMOUNT")))   # String → Decimal
        .withColumn("order_total",            F.col("ORDER_TOTAL").cast("decimal(38,4)"))
        # ----------------------------------------------------------------
        # Bronze-auditkolommen — ongewijzigd doorgegeven
        # ----------------------------------------------------------------
        .withColumn("_ingestion_timestamp",  F.col("_ingestion_timestamp"))
        .withColumn("_source_system",        F.col("_source_system"))
        .withColumn("_source_file",          F.col("_source_file"))
        .withColumn("_last_modified",        F.col("_last_modified"))
        .withColumn("_pipeline_run_id",      F.col("_pipeline_run_id"))
        # ----------------------------------------------------------------
        # Selecteer alleen de Silver-kolommen (laat alle uppercase Bronze-kolommen vallen
        # en behoud CDF-metadata voor apply_changes)
        # ----------------------------------------------------------------
        .select(
            # CDF-routerings-metadata
            "_change_type",
            "_commit_version",
            "_commit_timestamp",
            # Bedrijfskolommen (snake_case, type-fixed)
            "order_id",
            "truck_id",
            "location_id",
            "customer_id",
            "discount_id",
            "shift_id",
            "shift_start_time",
            "shift_end_time",
            "order_channel",
            "order_ts",
            "served_ts",
            "order_currency",
            "order_amount",
            "order_tax_amount",
            "order_discount_amount",
            "order_total",
            # Auditkolommen (Bronze-passthrough)
            "_ingestion_timestamp",
            "_source_system",
            "_source_file",
            "_last_modified",
            "_pipeline_run_id",
        )
    )

In [ ]:
@dlt.view(name="order_header_clean")
@dlt.expect_all(ORDER_HEADER_WARN_RULES)
def order_header_clean():
    """Cleansed-tak — rijen die elke drop-regel passeren.

    Filtert de type-fixed Bronze CDF-view op clean_predicate. Warn-regel-schendingen
    op rijen die dit filter overleven verschijnen in het DLT-eventlog via @expect_all
    (geen rijen worden gedropt). Fail-regels zijn gekoppeld aan `order_header_bronze_cdf`
    zodat ze de pipeline halten ongeacht de tak.
    """
    return dlt.read_stream("order_header_bronze_cdf").filter(ORDER_HEADER_CLEAN_PREDICATE)

In [ ]:
dlt.apply_changes(
    target="order_header",
    source="order_header_clean",
    keys=["order_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## order_header_quarantine — Patroon A inverse-filter-sink

Dezelfde structuur als de cleansed-tak, maar de bron-view filtert op `NOT (clean_predicate)`
en voegt een `failed_rules ARRAY<STRING>`-kolom toe die elke drop-regel die de rij schond
opsomt.  Beide takken worden gevoed door dezelfde Bronze CDF-view, dus de DLT-graph
toont ze als zustertabellen.

Voorbeeld van analyst-triage:
```sql
SELECT *
FROM   INTEGRATION.order_header_quarantine
WHERE  array_contains(failed_rules, 'order_total_non_negative');
```

In [ ]:
dlt.create_streaming_table(
    name="order_header_quarantine",
    comment="Quarantine — order_header rows that failed at least one drop rule. failed_rules lists which.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_header_quarantine_src")
def order_header_quarantine_src():
    """Quarantine-tak — rijen die ten minste één drop-regel schenden.

    Leest de type-fixed Bronze CDF-view, filtert op de inverse van clean_predicate,
    en voegt `failed_rules ARRAY<STRING>` toe met elke drop-regel die de rij schond.
    """
    return (
        dlt.read_stream("order_header_bronze_cdf")
        .filter(f"NOT ({ORDER_HEADER_CLEAN_PREDICATE})")
        .withColumn("failed_rules", build_failed_rules_expr(ORDER_HEADER_DROP_RULES))
    )

In [ ]:
dlt.apply_changes(
    target="order_header_quarantine",
    source="order_header_quarantine_src",
    keys=["order_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## Kwaliteitsregels — order_detail

Drie ernstniveaus, hetzelfde patroon als order_header:

- **fail** — `@expect_all_or_fail`; pipeline halt op schending. Gekoppeld aan de
  type-fixed Bronze CDF-view.
- **drop** — rij wordt uit de cleansed-tak gefilterd en geroute naar
  `order_detail_quarantine` met een `failed_rules`-array.
- **warn** — `@expect_all`; rij blijft in cleansed, schending verschijnt in DLT-
  event-metrics.

Regelnamen volgen het `<column>_<predicate>`-patroon zodat `failed_rules` zelfbeschrijvend is.

In [ ]:
ORDER_DETAIL_FAIL_RULES = {
    "order_detail_id_not_null": "order_detail_id IS NOT NULL",
}

ORDER_DETAIL_DROP_RULES = {
    "order_id_not_null":        "order_id IS NOT NULL",
    "menu_item_id_not_null":    "menu_item_id IS NOT NULL",
    "quantity_positive":        "quantity > 0",
    "unit_price_non_negative":  "unit_price >= 0",
    "price_non_negative":       "price >= 0",
}

ORDER_DETAIL_WARN_RULES = {
    "line_number_positive":     "line_number > 0",
}

ORDER_DETAIL_CLEAN_PREDICATE = build_clean_predicate(ORDER_DETAIL_DROP_RULES)

print(f"clean-predicaat: {ORDER_DETAIL_CLEAN_PREDICATE}")

## order_detail — gecleande Streaming Table

**Patroon:** Identiek aan order_header — Bronze CDF-view met type-fixes, clean-tak
gefilterd op het drop-regel-predicaat, quarantine-tak met inverse filter + `failed_rules`.

Type-fix voor order_detail:
- `ORDER_ITEM_DISCOUNT_AMOUNT` (StringType) → `order_item_discount_amount` (DecimalType(38,4))

Alle andere kolommen: zelfde type, hernoemd naar snake_case.

In [ ]:
dlt.create_streaming_table(
    name="order_detail",
    comment="Cleansed order_detail: type-fix ORDER_ITEM_DISCOUNT_AMOUNT→decimal(38,4), snake_case columns, CDF+apply_changes from Bronze.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_detail_bronze_cdf")
@dlt.expect_all_or_fail(ORDER_DETAIL_FAIL_RULES)
def order_detail_bronze_cdf():
    """Lees Bronze order_detail via CDF en pas type-fixes + snake_case-rename toe.

    Geeft een streaming DataFrame terug met:
    - Alle bedrijfskolommen hernoemd naar snake_case met type-correcties
    - Alle vijf Bronze-auditkolommen ongewijzigd doorgegeven
    - CDF-metadatakolommen (_change_type, _commit_version, _commit_timestamp)
      bewaard voor apply_changes-routering

    De fail-regel-Expectation halt de pipeline op `order_detail_id IS NULL`, ongeacht
    in welke downstream-tak (clean / quarantine) de rij anders zou belanden.
    """
    src = read_bronze_cdf(f"{catalog}.{bronze_schema}.order_detail")

    return (
        src
        # ----------------------------------------------------------------
        # Bedrijfskolommen — type-fix + snake_case
        # ----------------------------------------------------------------
        .withColumn("order_detail_id",            F.col("ORDER_DETAIL_ID").cast("decimal(38,0)"))
        .withColumn("order_id",                   F.col("ORDER_ID").cast("decimal(38,0)"))
        .withColumn("menu_item_id",               F.col("MENU_ITEM_ID").cast("decimal(38,0)"))
        .withColumn("quantity",                   F.col("QUANTITY").cast("decimal(38,4)"))
        .withColumn("unit_price",                 F.col("UNIT_PRICE").cast("decimal(38,4)"))
        .withColumn("price",                      F.col("PRICE").cast("decimal(38,4)"))
        .withColumn("order_item_discount_amount", cast_decimal_38_4(F.col("ORDER_ITEM_DISCOUNT_AMOUNT")))  # String → Decimal
        .withColumn("line_number",                F.col("LINE_NUMBER").cast("decimal(38,0)"))
        # ----------------------------------------------------------------
        # Bronze-auditkolommen — ongewijzigd doorgegeven
        # ----------------------------------------------------------------
        .withColumn("_ingestion_timestamp",  F.col("_ingestion_timestamp"))
        .withColumn("_source_system",        F.col("_source_system"))
        .withColumn("_source_file",          F.col("_source_file"))
        .withColumn("_last_modified",        F.col("_last_modified"))
        .withColumn("_pipeline_run_id",      F.col("_pipeline_run_id"))
        # ----------------------------------------------------------------
        # Selecteer alleen de Silver-kolommen (laat alle uppercase Bronze-kolommen vallen
        # en behoud CDF-metadata voor apply_changes)
        # ----------------------------------------------------------------
        .select(
            # CDF-routerings-metadata
            "_change_type",
            "_commit_version",
            "_commit_timestamp",
            # Bedrijfskolommen (snake_case, type-fixed)
            "order_detail_id",
            "order_id",
            "menu_item_id",
            "quantity",
            "unit_price",
            "price",
            "order_item_discount_amount",
            "line_number",
            # Auditkolommen (Bronze-passthrough)
            "_ingestion_timestamp",
            "_source_system",
            "_source_file",
            "_last_modified",
            "_pipeline_run_id",
        )
    )

In [ ]:
@dlt.view(name="order_detail_clean")
@dlt.expect_all(ORDER_DETAIL_WARN_RULES)
def order_detail_clean():
    """Cleansed-tak — rijen die elke drop-regel passeren.

    Filtert de type-fixed Bronze CDF-view op clean_predicate. Warn-regel-schendingen
    op rijen die dit filter overleven verschijnen in het DLT-eventlog via @expect_all
    (geen rijen worden gedropt). Fail-regels zijn gekoppeld aan `order_detail_bronze_cdf`
    zodat ze de pipeline halten ongeacht de tak.
    """
    return dlt.read_stream("order_detail_bronze_cdf").filter(ORDER_DETAIL_CLEAN_PREDICATE)

In [ ]:
dlt.apply_changes(
    target="order_detail",
    source="order_detail_clean",
    keys=["order_detail_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## order_detail_quarantine — Patroon A inverse-filter-sink

Dezelfde structuur als de cleansed-tak, maar de bron-view filtert op `NOT (clean_predicate)`
en voegt een `failed_rules ARRAY<STRING>`-kolom toe die elke drop-regel die de rij schond
opsomt.  Beide takken worden gevoed door dezelfde `order_detail_bronze_cdf`-view.

Voorbeeld van analyst-triage:
```sql
SELECT *
FROM   INTEGRATION.order_detail_quarantine
WHERE  array_contains(failed_rules, 'quantity_positive');
```

In [ ]:
dlt.create_streaming_table(
    name="order_detail_quarantine",
    comment="Quarantine — order_detail rows that failed at least one drop rule. failed_rules lists which.",
    table_properties={
        "delta.enableChangeDataFeed": "true",
    },
)

In [ ]:
@dlt.view(name="order_detail_quarantine_src")
def order_detail_quarantine_src():
    """Quarantine-tak — rijen die ten minste één drop-regel schenden.

    Leest de type-fixed Bronze CDF-view, filtert op de inverse van clean_predicate,
    en voegt `failed_rules ARRAY<STRING>` toe met elke drop-regel die de rij schond.
    """
    return (
        dlt.read_stream("order_detail_bronze_cdf")
        .filter(f"NOT ({ORDER_DETAIL_CLEAN_PREDICATE})")
        .withColumn("failed_rules", build_failed_rules_expr(ORDER_DETAIL_DROP_RULES))
    )

In [ ]:
dlt.apply_changes(
    target="order_detail_quarantine",
    source="order_detail_quarantine_src",
    keys=["order_detail_id"],
    sequence_by="_commit_version",
    apply_as_deletes=F.expr("_change_type = 'delete'"),
    apply_as_truncates=F.expr("_change_type = 'truncate'"),
    except_column_list=["_change_type", "_commit_version", "_commit_timestamp"],
    stored_as_scd_type=1,
)

## sales_line — Materialised View (geïntegreerde businessview)

`INTEGRATION.sales_line` is de geïntegreerde businessview: `order_header ⨝ order_detail`
gejoint op `order_id`, op regelgrain (één rij per order_detail-rij), met alle
order_header-attributen gedenormaliseerd op elke regel.

**Patroon:** Gedefinieerd als `@dlt.table` lezend via `spark.read` (batch, MV-semantiek)
vanuit de gecleande Silver Streaming Tables.  Quarantined rijen worden per definitie
uitgesloten omdat ze ontbreken in de gecleande input.

**Waarom MV, geen stream-stream-join?**
- Joins zijn eenvoudiger in batch — geen watermarks, geen state-management.
- Late aankomsten lossen zichzelf op bij de volgende pipeline-refresh.
- Header-updates (bijv. een gecorrigeerde `order_total`) propageren automatisch naar
  bestaande `sales_line`-rijen bij de volgende refresh — de MV wordt elke run volledig
  herberekend.

**Prefix-regel voor join-ambiguïteit:**  `order_header`-kolommen die een naam delen
met `order_detail` (alleen `order_id`) worden via het expliciete join-sleutel-predicaat
opgelost; `order_detail.order_id` wordt uit de output verwijderd — de join-sleutel is
de `order_header.order_id`-kopie.

In [ ]:
@dlt.table(
    name="sales_line",
    comment=(
        "Integrated business view: order_header ⨝ order_detail on order_id, "
        "one row per order line with all header attributes denormalised. "
        "Materialised View — batch spark.read, header updates propagate on next refresh."
    ),
)
def sales_line():
    """Materialised View — order_header ⨝ order_detail op regelgrain.

    Leest de gecleande Silver Streaming Tables via spark.read (batch-semantiek).
    DLT behandelt een @dlt.table die spark.read gebruikt als een Materialised View:
    deze wordt elke pipeline-run volledig herberekend, zodat laat aankomende detail-rijen
    en header-correcties beide automatisch oplossen zonder extra CDC-handling.

    Grain: één rij per order_detail-rij (order_detail_id is de unieke sleutel).
    Alle order_header-kolommen worden op elke regel gedenormaliseerd.
    Quarantined rijen worden uitgesloten omdat ze ontbreken in de gecleande input.
    """
    silver_schema = "INTEGRATION"

    oh = spark.read.table(f"{catalog}.{silver_schema}.order_header")
    od = spark.read.table(f"{catalog}.{silver_schema}.order_detail")

    return (
        od.join(oh, on="order_id", how="inner")
        .select(
            # -----------------------------------------------------------------
            # Regelgrain-sleutel
            # -----------------------------------------------------------------
            od.order_detail_id,
            od.order_id,          # join-sleutel — behouden van order_detail-zijde
            # -----------------------------------------------------------------
            # order_detail-bedrijfskolommen
            # -----------------------------------------------------------------
            od.menu_item_id,
            od.quantity,
            od.unit_price,
            od.price,
            od.order_item_discount_amount,
            od.line_number,
            # -----------------------------------------------------------------
            # order_header-bedrijfskolommen (gedenormaliseerd op elke regel)
            # -----------------------------------------------------------------
            oh.truck_id,
            oh.location_id,
            oh.customer_id,
            oh.discount_id,
            oh.shift_id,
            oh.shift_start_time,
            oh.shift_end_time,
            oh.order_channel,
            oh.order_ts,
            oh.served_ts,
            oh.order_currency,
            oh.order_amount,
            oh.order_tax_amount,
            oh.order_discount_amount,
            oh.order_total,
        )
    )

## Demo-notities

- **DLT-graph view**: De Databricks-UI toont vijf nodes:
  `order_header` + `order_header_quarantine` (uit `order_header_bronze_cdf`),
  `order_detail` + `order_detail_quarantine` (uit `order_detail_bronze_cdf`), en
  `sales_line` lezend uit zowel `order_header` als `order_detail` (alleen clean-tabellen).
- **Mode-switch test**: Bronze `order_detail` switchen tussen `full` en `incremental`
  en de workflow opnieuw draaien zal GEEN Silver-rijduplicatie of pipeline-falen
  veroorzaken.  Full-load-overwrites genereren `delete_row` + `insert_row`-events in
  CDF, en `apply_changes` (SCD Type 1 MERGE) houdt Silver consistent.
- **MV-propagatie demo**: Werk een `order_header`-rij bij (bijv. corrigeer een `order_total`).
  Na de volgende pipeline-run reflecteren de bijbehorende `sales_line`-rijen de gecorrigeerde
  waarde — de MV wordt elke refresh volledig herberekend.
- **Fail demo**: Voeg een Bronze-rij toe met `ORDER_DETAIL_ID = NULL` — `expect_all_or_fail`
  halt de pipeline.
- **Drop demo**: Voeg een Bronze-rij toe met `QUANTITY = 0`. Na de run verschijnt de rij
  in `INTEGRATION.order_detail_quarantine` met
  `array_contains(failed_rules, 'quantity_positive') = true`, afwezig in
  `INTEGRATION.order_detail` en daarmee ook afwezig in `INTEGRATION.sales_line`.
- **Warn demo**: Voeg een Bronze-rij toe met `LINE_NUMBER = 0`. De rij blijft in de
  cleansed-tabel; het schendings-aantal verschijnt in het DLT-eventlog.
- **Geen duplicatie**: Zowel header- als detail-quarantine-routering gebruiken
  `build_clean_predicate` en `build_failed_rules_expr` uit `_lib/rule_engine` — nul
  copy-paste-logica.